# Version 2: direction classification with an auxiliary return forecast
Upload the revised `train.py` and `inference.py` to `/home/sagemaker-user/stock-ml/src/` before running. Run training first and deploy only its successful new artifact. CSV features must already use the original training scaler. LONG/CASH now uses validation-selected positive-return probability, not the sign of the auxiliary return forecast. Existing Lambda signal handling remains compatible.
Validation must contain both signals and balanced accuracy >= 0.505. A failed gate means this run is not suitable for deployment; do not reuse an old model. Evaluate the frozen model on the full untouched test period after selection.


In [ ]:
from sagemaker.train import ModelTrainer
from sagemaker.core.training.configs import SourceCode, Compute, InputData
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core import image_uris

import boto3
import os
import shutil

In [ ]:
base_session = Session()

session = Session(
    default_bucket=base_session.default_bucket(),
    default_bucket_prefix="stock-ml"
)

role = get_execution_role()
region = session.boto_region_name

print("Region:", region)
print("Role:", role)

In [ ]:
bucket = "stock-dl-2026"
prefix = "processed/ml/"

s3 = boto3.client("s3", region_name=region)

response = s3.list_objects_v2(
    Bucket=bucket,
    Prefix=prefix
)

keys = [obj["Key"] for obj in response.get("Contents", [])]

train_key = next(k for k in keys if k.endswith("train.csv"))
val_key = next(k for k in keys if k.endswith("val.csv"))

train_uri = f"s3://{bucket}/{train_key}"
val_uri = f"s3://{bucket}/{val_key}"

print("Train:", train_uri)
print("Validation:", val_uri)

In [ ]:
source_code = SourceCode(
    source_dir="/home/sagemaker-user/stock-ml/src",
    entry_script="train.py"
)

print("Using:", "/home/sagemaker-user/stock-ml/src/train.py")

In [ ]:
training_image = image_uris.retrieve(
    framework="pytorch",
    region=region,
    version="2.2.0",
    py_version="py310",
    instance_type="ml.m5.xlarge",
    image_scope="training"
)

train_data = InputData(
    channel_name="train",
    data_source=train_uri
)

validation_data = InputData(
    channel_name="val",
    data_source=val_uri
)

compute = Compute(
    instance_type="ml.m5.xlarge",
    instance_count=1
)

In [ ]:
trainer = ModelTrainer(
    training_image=training_image,
    role=role,
    sagemaker_session=session,
    source_code=source_code,
    compute=compute,
    input_data_config=[
        train_data,
        validation_data
    ],
    base_job_name="stock-ffnn-direction-v2",
    hyperparameters={
        "batch-size": 512,
        "epochs": 100,
        "learning-rate": 1e-3,
        "weight-decay": 1e-6,
        "patience": 10
    }
)

In [ ]:
trainer.train()

In [ ]:
job_name = trainer._latest_training_job.training_job_name
print("Training job:", job_name)

sm = boto3.client("sagemaker", region_name=region)

job = sm.describe_training_job(
    TrainingJobName=job_name
)

print("Status:", job["TrainingJobStatus"])
print("Model artifact:", job["ModelArtifacts"]["S3ModelArtifacts"])
print("Training seconds:", job["TrainingTimeInSeconds"])

In [ ]:
from urllib.parse import urlparse

model_uri = job["ModelArtifacts"]["S3ModelArtifacts"]

parsed = urlparse(model_uri)

source_bucket = parsed.netloc
source_key = parsed.path.lstrip("/")

destination_bucket = "stock-dl-2026"
destination_key = "models/ffnn-v2/model.tar.gz"

s3 = boto3.client("s3", region_name=region)

s3.copy_object(
    Bucket=destination_bucket,
    Key=destination_key,
    CopySource={
        "Bucket": source_bucket,
        "Key": source_key
    }
)

print(
    "Copied to:",
    f"s3://{destination_bucket}/{destination_key}"
)

In [ ]:
job_name = trainer._latest_training_job.training_job_name

logs = boto3.client("logs", region_name=region)

streams = logs.describe_log_streams(
    logGroupName="/aws/sagemaker/TrainingJobs",
    logStreamNamePrefix=job_name
)

stream_name = streams["logStreams"][0]["logStreamName"]

events = logs.get_log_events(
    logGroupName="/aws/sagemaker/TrainingJobs",
    logStreamName=stream_name,
    startFromHead=True
)

for event in events["events"]:
    message = event["message"]
    if (
        "Epoch" in message
        or "Best validation loss" in message
        or "Early stopping" in message
    ):
        print(message)

In [ ]:
import tarfile

bucket = "stock-dl-2026"

model_key = "models/ffnn-v2/model.tar.gz"
local_model_tar = "/tmp/model.tar.gz"
extract_dir = "/tmp/model_package"
new_tar = "/tmp/model-inference.tar.gz"

s3 = boto3.client("s3", region_name=region)

# Download trained model
s3.download_file(
    bucket,
    model_key,
    local_model_tar
)

# Extract it
shutil.rmtree(extract_dir, ignore_errors=True)
os.makedirs(extract_dir, exist_ok=True)

with tarfile.open(local_model_tar, "r:gz") as tar:
    tar.extractall(extract_dir)

# Add inference.py
code_dir = os.path.join(extract_dir, "code")
os.makedirs(code_dir, exist_ok=True)

shutil.copy2(
    "/home/sagemaker-user/stock-ml/src/inference.py",
    os.path.join(code_dir, "inference.py")
)

# Repack
with tarfile.open(new_tar, "w:gz") as tar:
    for name in os.listdir(extract_dir):
        tar.add(
            os.path.join(extract_dir, name),
            arcname=name
        )

print("Package contents:")

for root, dirs, files in os.walk(extract_dir):
    for file in files:
        print(os.path.relpath(
            os.path.join(root, file),
            extract_dir
        ))

In [ ]:
inference_model_key = "models/ffnn-v2/model-inference.tar.gz"

s3.upload_file(
    new_tar,
    bucket,
    inference_model_key
)

model_data_uri = (
    f"s3://{bucket}/{inference_model_key}"
)

print(model_data_uri)

In [ ]:
inference_image = image_uris.retrieve(
    framework="pytorch",
    region=region,
    version="2.2.0",
    py_version="py310",
    instance_type="ml.m5.xlarge",
    image_scope="inference"
)

print(inference_image)